<a href="https://colab.research.google.com/github/JeffersonRodrigues9/Automacao_com_python/blob/main/Automa%C3%A7%C3%A3o_Dowload_S3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Automação desenvolvida para buscar e baixar arquivos no Amazon S3
# a partir de uma lista de números contida em um arquivo TXT.
# Neste projeto utilizei Python e a biblioteca boto3 para percorrer
# estruturas de pastas no bucket, identificar correspondências e
# realizar o download automático dos arquivos encontrados.

import boto3
from pathlib import Path

BUCKET_NOME = ""
PREFIXO = ""

ARQUIVO_LISTA = r".txt"
PASTA_DESTINO = r""

s3 = boto3.client(
    "s3",
    aws_access_key_id="",
    aws_secret_access_key=""
)

def ler_lista():
    with open(ARQUIVO_LISTA, "r", encoding="utf-8") as f:
        return set(l.strip().lstrip("0") for l in f if l.strip())

def listar_tudo():
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NOME, Prefix=PREFIXO):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if not key.endswith("/"):
                yield key

def baixar(s3_key):
    caminho_local = Path(PASTA_DESTINO) / s3_key
    caminho_local.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(BUCKET_NOME, s3_key, str(caminho_local))

def processar():
    numeros = ler_lista()

    print("\nEntrando em lastro/2025\n")

    ultimo_mes = None
    ultimo_dia = None
    ultimo_numero = None

    for key in listar_tudo():
        partes = key.split("/")

        if len(partes) < 6:
            continue

        mes = partes[2]
        dia = partes[3]
        numero = partes[4]

        if mes != ultimo_mes:
            print(f"\nMês: {mes}")
            ultimo_mes = mes

        if dia != ultimo_dia:
            print(f"   Dia: {dia}")
            ultimo_dia = dia

        if numero != ultimo_numero:
            print(f"      Verificando: {numero}", end="")

            if numero.lstrip("0") in numeros:
                print("  MATCH!")
                baixar(key)
            else:
                print("  NÃO ENCONTRADO")

            ultimo_numero = numero
        else:
            # mesma pasta → só baixa se já deu match
            if numero.lstrip("0") in numeros:
                baixar(key)

if __name__ == "__main__":
    processar()